# Plot spot position as time series data
env : data_vis_32

# 1.0 Import relevant packages

In [1]:
# import pypyodbc
import pandas as pd
import plotly.express as px
import seaborn as sns
from matplotlib.colors import to_hex
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import numpy as np

# 2.0 Import spot position and size QA data

In [2]:
data_path = r"../data/xlsx_exported_from_access/SpotPositionResults.xlsx"

df = pd.read_excel(data_path)

df.head(2)



,ADate,MachineName,Energy,Device,Gantry Angle,Spot,x-pos,y-pos,hor_rt_gradient,hor_lt_gradient,hor_fwhm,vert_rt_gradient,vert_lt_gradient,vert_fwhm,bltr_rt_gradient,bltr_lt_gradient,bltr_fwhm,tlbr_rt_gradient,tlbr_lt_gradient,tlbr_fwhm
0,2022-04-21 16:44:14,Gantry 2,70,XRV-3000,180,Bottom-Centre,-0.2357,124.9224,-9.059840,9.157258,13.342140,-8.964427,9.333333,13.536155,-8.485281,8.747554,14.216962,-8.909545,8.992812,13.842831
1,2022-04-21 16:44:14,Gantry 2,70,XRV-3000,180,Bottom-Left,-125.0153,125.4501,-8.871094,9.120482,13.562307,-8.964427,9.333333,13.712522,-8.747554,8.591347,14.310494,-8.747554,8.992812,13.842831


# 3.0 exploratory data analysis - understand your data

In [3]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41172 entries, 0 to 41171
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   ADate             41172 non-null  datetime64[ns]
 1   MachineName       41172 non-null  object        
 2   Energy            41172 non-null  int64         
 3   Device            41172 non-null  object        
 4   Gantry Angle      41172 non-null  int64         
 5   Spot              41172 non-null  object        
 6   x-pos             41172 non-null  float64       
 7   y-pos             41172 non-null  float64       
 8   hor_rt_gradient   41172 non-null  float64       
 9   hor_lt_gradient   41172 non-null  float64       
 10  hor_fwhm          41172 non-null  float64       
 11  vert_rt_gradient  41172 non-null  float64       
 12  vert_lt_gradient  41172 non-null  float64       
 13  vert_fwhm         41172 non-null  float64       
 14  bltr_rt_gradient  4117

In [4]:
df.value_counts("MachineName"), df.value_counts("Device"), df.value_counts("Energy")

(MachineName
 Gantry 3    10576
 Gantry 1    10472
 Gantry 4    10357
 Gantry 2     9767
 Name: count, dtype: int64,
 Device
 XRV-3000    31815
 XRV-4000     9357
 Name: count, dtype: int64,
 Energy
 150    8250
 240    8243
 200    8233
 100    8232
 70     8214
 Name: count, dtype: int64)

# 4.0 filtering ABS Shift data

In [5]:
sub_df = df[["ADate",	"MachineName", 	"Energy", "Device", "Gantry Angle", "Spot", "x-pos", "y-pos"]].copy()

## calculate abs shift

In [6]:
pred_xrv4000 = {'Top-Top-Left': [-125, -175], 'Top-Top-Centre': [0, -175], 'Top-Top-Right': [125, -175], \
                'Top-Left': [-125, -125], 'Top-Centre':[0, -125], 'Top-Right':[125, -125], \
                'Left': [-125, 0], 'Centre':[0, 0], 'Right':[125, 0], \
                'Bottom-Left': [-125, 125], 'Bottom-Centre':[0, 125], 'Bottom-Right':[125, 125], \
                'Bottom-Bottom-Left': [-125, 175], 'Bottom-Bottom-Centre': [0, 175], 'Bottom-Bottom-Right': [125, 175]}

sub_df['px_pos'] = sub_df['Spot'].map(lambda s: pred_xrv4000[s][0] if s in pred_xrv4000 else None)
sub_df['py_pos'] = sub_df['Spot'].map(lambda s: pred_xrv4000[s][1] if s in pred_xrv4000 else None)

sub_df['abs_xpos'] = sub_df["x-pos"] - sub_df["px_pos"]
sub_df['abs_ypos'] = sub_df["y-pos"] - sub_df["py_pos"]

In [9]:
print(sub_df.head(2))

                ADate MachineName  Energy    Device  Gantry Angle  \
0 2022-04-21 16:44:14    Gantry 2      70  XRV-3000           180   
1 2022-04-21 16:44:14    Gantry 2      70  XRV-3000           180   

            Spot     x-pos     y-pos  px_pos  py_pos  abs_xpos  abs_ypos  
0  Bottom-Centre   -0.2357  124.9224       0     125   -0.2357   -0.0776  
1    Bottom-Left -125.0153  125.4501    -125     125   -0.0153    0.4501  


In [8]:

def plotly_spot_position(df, pos, gantry, device, energy, gantry_angle, n_months):
    """ plot spot position time series data
        df = dataframe
        gantry = "Gantry 1", "Gantry 2", 
        pos = "abs_xpos",
        device = "XRV-3000", "XRV-4000"
        energy = int,
        gantry_angle = 0,90,180,270
        n_month = int
     """
    
     # only show data from last 12 months
    start_date = pd.Timestamp.today() - pd.DateOffset(months=n_months)

    selected_df = df[(df["MachineName"]==gantry) & (df["Device"] == device) & (df['ADate'] >= start_date) & (df["Energy"] == energy) &(df["Gantry Angle"] == gantry_angle)]

    # set colour
    palette = sns.color_palette("deep", n_colors=df['Spot'].nunique())
    palette_hex = [to_hex(c) for c in palette]


    # Plot
    fig = px.scatter(
        selected_df,
        x='ADate',
        y=pos,
        symbol='Spot', 
        color='Spot',        # hue
         color_discrete_sequence= px.colors.qualitative.T10,
        title=f'{gantry} - absolute shift- {pos}',
        labels={'x-pos': 'X Position', 'adate': 'Date'},
        height=500
    )

    # Add tolerance bands +/- 2
    fig.add_hline(y=2, line_dash="dash", line_color="grey", annotation_text="tolerance", annotation_position="top left")
    fig.add_hline(y=-2, line_dash="dash", line_color="grey")



    # Optional: connect points by spot for clarity
    fig.update_traces(mode='markers+lines',
                    marker=dict(size=12,               # larger size
                                line=dict(width=2)),     # outline width
                    line=dict(width=1                # thinner connecting lines
                    ))

    # Show plot
    fig.show()

    
    return 


# plotting absolute y-pos, on Gantry 4, from XRV-4000 data, 70 MeV spot, Gantry angle = 0, in last2 months
plotly_spot_position(sub_df, "abs_ypos", "Gantry 4", "XRV-4000", 70, 0, 24)


In [9]:
plotly_spot_position(sub_df, "abs_ypos", "Gantry 4", "XRV-4000", 70, 0, 24)

In [5]:
start_date = pd.Timestamp.today() - pd.DateOffset(months=12)
selected_df = sub_df[(df["MachineName"]=="Gantry 2") & (df["Device"] == "XRV-3000") & (df['ADate'] >= start_date)].copy()
# Calculate average abs_xpos per adate and energy
selected_df['avg_abs_pos'] = selected_df.groupby(['ADate', 'Energy', "Gantry Angle"])["abs_xpos"].transform('mean')

selected_df.head(5)

,ADate,MachineName,Energy,Device,Gantry Angle,Spot,x-pos,y-pos,px_pos,py_pos,abs_xpos,abs_ypos,avg_abs_pos
34149,2025-06-21 09:49:27,Gantry 2,70,XRV-3000,0,Bottom-Centre,0.4379,125.3434,0,125,0.4379,0.3434,0.407056
34150,2025-06-21 09:49:27,Gantry 2,70,XRV-3000,0,Bottom-Left,-124.6581,125.3003,-125,125,0.3419,0.3003,0.407056
34151,2025-06-21 09:49:27,Gantry 2,70,XRV-3000,0,Bottom-Right,125.6795,125.2833,125,125,0.6795,0.2833,0.407056
34152,2025-06-21 09:49:27,Gantry 2,70,XRV-3000,0,Centre,0.3853,0.3546,0,0,0.3853,0.3546,0.407056
34153,2025-06-21 09:49:27,Gantry 2,70,XRV-3000,0,Left,-124.8079,0.7124,-125,0,0.1921,0.7124,0.407056


In [11]:

def plotly_ave_spot_position(df, parameter, gantry, device,  n_months):
    """ plot average spot position across all spot positions with the same adate and energy
        df = dataframe
        gantry = "Gantry 1", "Gantry 2", 
        paramter = "abs_xpos",
        device = "XRV-3000", "XRV-4000"
        energy = int
        n_month = int

    
     """
    
     # only show data from last 12 months
    start_date = pd.Timestamp.today() - pd.DateOffset(months=n_months)

    selected_df = df[(df["MachineName"]==gantry) & (df["Device"] == device) & (df['ADate'] >= start_date)].copy()


    # Calculate average abs_xpos per adate and energy
    selected_df['avg_abs_pos'] = selected_df.groupby(['ADate', 'Energy', "Gantry Angle"])[parameter].transform('mean')

    # want to displace energy as discrete colour not spectrum
    selected_df['Energy'] = df['Energy'].astype(int).astype(str)

    # Plot
    fig = px.scatter(
        selected_df,
        x='ADate',
        y='avg_abs_pos',
        symbol='Gantry Angle', 
        color='Energy',        # hue
        title=f'average {parameter} across all spot positions with the same adate and energy',
        labels={'x-pos': 'X Position', 'adate': 'Date'},
        height=500
    )

    # Add tolerance bands +/- 2
    fig.add_hline(y=2, line_dash="dash", line_color="grey", annotation_text="tolerance", annotation_position="top left")
    fig.add_hline(y=-2, line_dash="dash", line_color="grey")



    # Optional: connect points by spot for clarity
       # Optional: connect points by spot for clarity
    fig.update_traces(mode='markers+lines',
                    marker=dict(size=12,               # larger size
                                line=dict(width=2)),     # outline width
                    line=dict(width=1                # thinner connecting lines
                    ))

    # Show plot
    fig.show()

    
    return 




## Demonstrate drop down filters

### Single graph (yshift)

In [12]:
# Plotting Function
def plotly_spot_position_Filtered(df, pos, gantry, device, energy, gantry_angle, n_months):
    """ plot spot position time series data
        df = dataframe
        gantry = "Gantry 1", "Gantry 2", 
        pos = "abs_xpos",
        device = "XRV-3000", "XRV-4000"
        energy = int,
        gantry_angle = 0,90,180,270
        n_month = int
     """
    
     # only show data from last 12 months
    start_date = pd.Timestamp.today() - pd.DateOffset(months=n_months)

    selected_df = df[(df["MachineName"]==gantry) & (df["Device"] == device) & (df['ADate'] >= start_date) & (df["Energy"] == energy) &(df["Gantry Angle"] == gantry_angle)]

    energies = sorted(selected_df["Energy"].dropna().unique())
    gantry_angles = sorted(selected_df["Gantry Angle"].dropna().unique())

    # set colour
    palette = sns.color_palette("deep", n_colors=df['Spot'].nunique())
    palette_hex = [to_hex(c) for c in palette]


    # Plot
    fig = px.scatter(
        selected_df,
        x='ADate',
        y=pos,
        symbol='Spot', 
        color='Spot',        # hue
         color_discrete_sequence= px.colors.qualitative.T10,
        title=f'Absolute shift - {pos}',
        labels={'x-pos': 'X Position', 'adate': 'Date'},
        height=500
    )

    # Add tolerance bands +/- 2
    fig.add_hline(y=2, line_dash="dash", line_color="grey", annotation_text="tolerance", annotation_position="top left")
    fig.add_hline(y=-2, line_dash="dash", line_color="grey")

    # Optional: connect points by spot for clarity
    fig.update_traces(mode='markers+lines',
                    marker=dict(size=12,               # larger size
                                line=dict(width=2)),     # outline width
                    line=dict(width=1                # thinner connecting lines
                    ))
    
    # Update to include drop down box for filtering
    
    # Functions to enable automatically updating graph filtering from drop-down menus
    def visible_for_g(target_g): # Gantry selection
        return [
            True if g == target_g else "legendonly"
            for g in ["Gantry 1", "Gantry 2", "Gantry 3", "Gantry 4"]
        ]
    
    def visible_for_ga(target_ga): # Gantry Angle selection
        return [
            True if ga == target_ga else "legendonly"
            for ga in ["0", "90", "180", "270"]
        ]
    
    def visible_for_e(target_e): # Energy selection
        return [
            True if e == target_e else "legendonly"
            for e in ["70", "100", "150", "200", "270"]
        ]

    # Define Gantry Drop-down menu
    g_button = []
    for g in ["Gantry 1", "Gantry 2", "Gantry 3", "Gantry 4"]:
            g_button.append(
                dict(
                    label=f"{g}",
                    method="update",
                    args=[{"visible": visible_for_g(g)}]
                    )
            )

    # Define Gantry Angle drop-down menu
    ga_button = []
    for ga in ["0", "90", "180", "270"]:
            ga_button.append(
                dict(
                    label=f"{ga}",
                    method="update",
                    args=[{"visible": visible_for_ga(ga)}]
                    )
            )

    # Define Energy drop-down menu
    e_button = []
    for e in ["70", "100", "150", "200", "270"]:
            e_button.append(
                dict(
                    label=f"{e} MeV",
                    method="update",
                    args=[{"visible": visible_for_e(e)}]
                    )
            )

    fig.update_layout(
        updatemenus=[
            # Update graph to reflect Gantry drop down box value
            dict(
                type="dropdown",
                direction="down",
                x=0.02,
                y=1.12,
                xanchor="left",
                yanchor="top",
                showactive=True,
                buttons=g_button
            ),

            # Update graph to reflect Gantry Angle drop down box value
            dict(
                type="dropdown",
                direction="down",
                x=0.16,
                y=1.12,
                xanchor="left",
                yanchor="top",
                showactive=True,
                buttons=ga_button
            ),

            # Update graph to reflect Energy drop down box value
            dict(
                type="dropdown",
                direction="down",
                x=0.27,
                y=1.12,
                xanchor="left",
                yanchor="top",
                showactive=True,
                buttons=e_button
            ),
        ]
    ),

    fig.update_layout(
        annotations=[
            dict(text="Gantry", x=0.0, xref="paper", y=1.1, yref="paper", align = "left", showarrow=False),
            dict(text="Angle:", x=0.12, xref="paper", y=1.1, yref="paper", align="left", showarrow=False),
            dict(text="Energy:", x=0.22, xref="paper", y=1.1, yref="paper", align="left", showarrow=False)
        ])


    # Show Plot
    fig.show()
    return 


Test plotting function

In [13]:
# plotting absolute y-pos, on Gantry 4, from XRV-4000 data, 70 MeV spot, Gantry angle = 0, in last2 months
plotly_spot_position_Filtered(sub_df, "abs_xpos", "Gantry 2", "XRV-3000", 70, 180, 60)
plotly_spot_position_Filtered(sub_df, "abs_ypos", "Gantry 2", "XRV-3000", 70, 180, 60)

### Xshift/Yshift Filter on same graph

In [14]:
# Plotting Function
def plotly_spot_position_Filtered_XY(df, gantry, device, energy, gantry_angle, n_months):
    """ plot spot position time series data
        df = dataframe
        gantry = "Gantry 1", "Gantry 2", 
        pos = "abs_xpos",
        device = "XRV-3000", "XRV-4000"
        energy = int,
        gantry_angle = 0,90,180,270
        n_month = int
     """
    
    # only show data from last 12 months
    start_date = pd.Timestamp.today() - pd.DateOffset(months=n_months)

    # set colour
    palette = sns.color_palette("deep", n_colors=df['Spot'].nunique())
    palette_hex = [to_hex(c) for c in palette]


    # Plot x_pos figure
    fig = px.scatter(
        df,
        x='ADate',
        y="abs_xpos",
        symbol='Spot', 
        color='Spot',        # hue
         color_discrete_sequence= px.colors.qualitative.T10,
        title='Absolute shift - Position',
        labels={'adate': 'Date'},
        height=500
    )

    # Add tolerance bands +/- 2
    fig.add_hline(y=2, line_dash="dash", line_color="grey", annotation_text="tolerance", annotation_position="top left")
    fig.add_hline(y=-2, line_dash="dash", line_color="grey")

    # Optional: connect points by spot for clarity
    fig.update_traces(mode='markers+lines',
                    marker=dict(size=12,               # larger size
                                line=dict(width=2)),     # outline width
                    line=dict(width=1                # thinner connecting lines
                    ))
    
    # Update to include drop down box for filtering
    # Functions to enable automatically updating graph filtering from drop-down menus
    def visible_for_g(target_g): # Gantry selection
        return [
            True if g == target_g else False
            for g in ["Gantry 1", "Gantry 2", "Gantry 3", "Gantry 4"]
        ]
    
    def visible_for_ga(target_ga): # Gantry Angle selection
        return [
            True if ga == target_ga else False

            for ga in ["0", "90", "180", "270"]
        ]
    
    def visible_for_e(target_e): # Energy selection
        return [
            True if e == target_e else False
            for e in ["70", "100", "150", "200", "270"]
        ]

    def visible_for_pos(target_pos): # Position selection
        return [
            True if pos == target_pos else False
            for pos in ["abs_xpos", "abs_ypos"]
        ]

    # Define Gantry Drop-down menu
    g_button = []
    for g in ["Gantry 1", "Gantry 2", "Gantry 3", "Gantry 4"]:
            g_button.append(
                dict(
                    label=f"{g}",
                    method="update",
                    args=[{"visible": visible_for_g(g)}]
                    )
            )

    # Define Gantry Angle drop-down menu
    ga_button = []
    for ga in ["0", "90", "180", "270"]:
            ga_button.append(
                dict(
                    label=f"{ga}",
                    method="update",
                    args=[{"visible": visible_for_ga(ga)}]
                    )
            )

    # Define Energy drop-down menu
    e_button = []
    for e in ["70", "100", "150", "200", "270"]:
            e_button.append(
                dict(
                    label=f"{e} MeV",
                    method="update",
                    args=[{"visible": visible_for_e(e)}]
                    )
            )

    # Define x/y_pos drop-down menu
    pos_button = []
    for pos in ["abs_xpos", "abs_ypos"]:
            pos_button.append(
                dict(
                    label=f"{pos}",
                    method="update",
                    args=[{"visible": visible_for_pos(pos)}]
                    )
            )

    fig.update_layout(
        updatemenus=[
            # Update graph to reflect Gantry drop down box value
            dict(
                type="dropdown",
                direction="down",
                x=0.02,
                y=1.12,
                xanchor="left",
                yanchor="top",
                showactive=True,
                buttons=g_button
            ),

            # Update graph to reflect Gantry Angle drop down box value
            dict(
                type="dropdown",
                direction="down",
                x=0.16,
                y=1.12,
                xanchor="left",
                yanchor="top",
                showactive=True,
                buttons=ga_button
            ),

            # Update graph to reflect Energy drop down box value
            dict(
                type="dropdown",
                direction="down",
                x=0.28,
                y=1.12,
                xanchor="left",
                yanchor="top",
                showactive=True,
                buttons=e_button
            ),

            # Update graph to reflect Position drop down box value
            dict(
                type="dropdown",
                direction="down",
                x=0.43,
                y=1.12,
                xanchor="left",
                yanchor="top",
                showactive=True,
                buttons=pos_button
            ),
        ]
    ),

    fig.update_layout(
        annotations=[
            dict(text="Gantry", x=0.0, xref="paper", y=1.1, yref="paper", align = "left", showarrow=False),
            dict(text="Angle:", x=0.12, xref="paper", y=1.1, yref="paper", align="left", showarrow=False),
            dict(text="Energy:", x=0.23, xref="paper", y=1.1, yref="paper", align="left", showarrow=False),
            dict(text="Position:", x=0.40, xref="paper", y=1.1, yref="paper", align="left", showarrow=False)
        ])

    # Show Plot
    fig.show()
    return 


Test plotting function

In [15]:
# plotting absolute both x-pos/y-pos shift, on Gantry 4, from XRV-4000 data, 70 MeV spot, Gantry angle = 0, in last2 months
plotly_spot_position_Filtered_XY(sub_df, "abs_ypos", "Gantry 4", "XRV-4000", 70, 0, 24)

TypeError: plotly_spot_position_Filtered_XY() takes 6 positional arguments but 7 were given

### Xshift/Yshift Filter on same graph CORRECTED

In [16]:
# Plotting Function
def plotly_spot_position_Filtered_XY(df, pos, gantry, device, energy, gantry_angle, n_months):
    """ plot spot position time series data
        df = spot position dataframe
     """
    
    # only show data from last 12 months
    start_date = pd.Timestamp.today() - pd.DateOffset(months=n_months)
    selected_df = df[(df["MachineName"]==gantry) & (df["Device"] == device) & (df['ADate'] >= start_date) & (df["Energy"] == energy) &(df["Gantry Angle"] == gantry_angle)]
    

    # set colour
    palette = sns.color_palette("deep", n_colors=df['Spot'].nunique())
    palette_hex = [to_hex(c) for c in palette]

    # Plot y_pos figure
    fig = px.scatter(
        selected_df,
        x='ADate',
        y="abs_ypos",
        symbol='Spot', 
        color='Spot', # hue
        color_discrete_sequence= px.colors.qualitative.T10,
        title='Absolute shift',
        labels={'adate': 'Date'},
        height=500
    )

    # Add tolerance bands +/- 2
    fig.add_hline(y=2, line_dash="dash", line_color="grey", annotation_text="tolerance", annotation_position="top left")
    fig.add_hline(y=-2, line_dash="dash", line_color="grey")

    # Optional: connect points by spot for clarity
    fig.update_traces(mode='markers+lines',
                    marker=dict(size=12,               # larger size
                                line=dict(width=2)),     # outline width
                    line=dict(width=1                # thinner connecting lines
                    ))
    
    # Update to include drop down box for filtering
    # Functions to enable automatically updating graph filtering from drop-down menus
    def visible_for_g(target_g): # Gantry selection
        return [
            True if g == target_g else "legendonly"
            for g in ["Gantry 1", "Gantry 2", "Gantry 3", "Gantry 4"]
        ]
    
    def visible_for_ga(target_ga): # Gantry Angle selection
        return [
            True if ga == target_ga else "legendonly"
            for ga in ["0", "90", "180", "270"]
        ]
    
    def visible_for_e(target_e): # Energy selection
        return [
            True if e == target_e else "legendonly"
            for e in ["70", "100", "150", "200", "270"]
        ]

    def visible_for_pos(target_pos): # Position selection
        return [
            True if pos == target_pos else "legendonly"
            for pos in ["abs_xpos", "abs_ypos"]
        ]

    # Define Gantry Drop-down menu
    g_button = []
    for g in ["Gantry 1", "Gantry 2", "Gantry 3", "Gantry 4"]:
            g_button.append(
                dict(
                    label=f"{g}",
                    method="update",
                    args=[{"visible": visible_for_g(g)}]
                    )
            )

    # Define Gantry Angle drop-down menu
    ga_button = []
    for ga in ["0", "90", "180", "270"]:
            ga_button.append(
                dict(
                    label=f"{ga}",
                    method="update",
                    args=[{"visible": visible_for_ga(ga)}]
                    )
            )

    # Define Energy drop-down menu
    e_button = []
    for e in ["70", "100", "150", "200", "270"]:
            e_button.append(
                dict(
                    label=f"{e} MeV",
                    method="update",
                    args=[{"visible": visible_for_e(e)}]
                    )
            )

    # Define x/y_pos drop-down menu
    pos_button = []
    for pos in ["abs_xpos", "abs_ypos"]:
            pos_button.append(
                dict(
                    label=f"{pos}",
                    method="update",
                    args=[{"visible": visible_for_pos(pos)}]
                    )
            )

    fig.update_layout(
        updatemenus=[
            # Update graph to reflect Gantry drop down box value
            dict(
                type="dropdown",
                direction="down",
                x=0.02,
                y=1.14,
                xanchor="left",
                yanchor="top",
                showactive=True,
                buttons=g_button
            ),

            # Update graph to reflect Gantry Angle drop down box value
            dict(
                type="dropdown",
                direction="down",
                x=0.16,
                y=1.14,
                xanchor="left",
                yanchor="top",
                showactive=True,
                buttons=ga_button
            ),

            # Update graph to reflect Energy drop down box value
            dict(
                type="dropdown",
                direction="down",
                x=0.28,
                y=1.14,
                xanchor="left",
                yanchor="top",
                showactive=True,
                buttons=e_button
            ),

            # Update graph to reflect Position drop down box value
            dict(
                type="dropdown",
                direction="down",
                x=0.43,
                y=1.14,
                xanchor="left",
                yanchor="top",
                showactive=True,
                buttons=pos_button
            ),
        ]
    ),

    fig.update_layout(
        annotations=[
            dict(text="Gantry", x=0.0, xref="paper", y=1.12, yref="paper", align = "left", showarrow=False),
            dict(text="Angle:", x=0.12, xref="paper", y=1.12, yref="paper", align="left", showarrow=False),
            dict(text="Energy:", x=0.23, xref="paper", y=1.12, yref="paper", align="left", showarrow=False),
            dict(text="Position:", x=0.40, xref="paper", y=1.1, yref="paper", align="left", showarrow=False)
        ])
    

    # Create and add slider
    months = []
    for i in range(n_months):
        month = dict(
            method="update",
            args=[{"visible": [True] * len(fig.data)}  # layout attribute
        ])
        month["args"][0]["visible"][i] = True  # Toggle i'th trace to "visible"
        months.append(month)

    # Define sliders for date range selection
    sliders = [dict(
    active=10,
    currentvalue={"prefix": "Previous "},
    pad={"t": 50},
    steps=months
    )]

    fig.update_layout(
        sliders=sliders
    )

    i=0
    for step in fig.layout.sliders[0].steps:
        step['label'] = f"{i} Months"
        i += 1

    # Show Plot
    fig.show()
    return 


In [17]:
# plotting absolute both x-pos/y-pos shift, on Gantry 4, from XRV-4000 data, 70 MeV spot, Gantry angle = 0, in last2 months
plotly_spot_position_Filtered_XY(sub_df, "abs_ypos", "Gantry 4", "XRV-4000", 70, 0, 12)

# Spatial Grid Subplot

## SKC

In [7]:
pred_xrv4000 = {'Top-Top-Left': [-125, -175], 'Top-Top-Centre': [0, -175], 'Top-Top-Right': [125, -175], \
                'Top-Left': [-125, -125], 'Top-Centre':[0, -125], 'Top-Right':[125, -125], \
                'Left': [-125, 0], 'Centre':[0, 0], 'Right':[125, 0], \
                'Bottom-Left': [-125, 125], 'Bottom-Centre':[0, 125], 'Bottom-Right':[125, 125], \
                'Bottom-Bottom-Left': [-125, 175], 'Bottom-Bottom-Centre': [0, 175], 'Bottom-Bottom-Right': [125, 175]}

pred_xrv3000 = {'Top-Left': [-125, -125], 'Top-Centre':[0, -125], 'Top-Right':[125, -125], \
                'Left': [-125, 0], 'Centre':[0, 0], 'Right':[125, 0], \
                'Bottom-Left': [-125, 125], 'Bottom-Centre':[0, 125], 'Bottom-Right':[125, 125]}

markers_plotly = {240: 'circle-open', 200: 'triangle-left', 150:'square', 100:'x-open', 70:'triangle-up'}

In [21]:

def plot_spot_grid_plotly(df, gantry, device, tolerance, n_months):
    """ plot the data in grids. """

    # 1. Data Preparation
    df['ADate'] = pd.to_datetime(df['ADate'])
    start_date = pd.Timestamp.today() - pd.DateOffset(months=n_months)
    selected_df = df[(df["MachineName"]==gantry) & (df["Device"] == device) & (df['ADate'] >= start_date)].copy()

    if selected_df.empty:
        print(f"No data found for {gantry} using {device} in the last {n_months} months.")
        return None

    # 2. Dynamic Color Palette (HUSL)
    # Sorting ensures the color gradient follows the energy levels
    ens = sorted(list(selected_df["Energy"].unique()), reverse=True)
    husl_palette = sns.color_palette("husl", len(ens)).as_hex()
    colours = dict(zip(ens, husl_palette))

    # 3. Plot Configuration
    if tolerance == 1:
        b = 2.5
        title = f"Relative spot positions ({tolerance} mm tolerance)"
    else:
        b = 5
        title = f"Absolute spot positions ({tolerance} mm tolerance)"
        
    if device == 'XRV-4000':
        pos = pred_xrv4000
        nrows, ncols = 5, 3
    elif device == 'XRV-3000':
        pos = pred_xrv3000
        nrows, ncols = 3, 3
    
    fig = make_subplots(rows=nrows, cols=ncols,
                        subplot_titles=list(pos.keys()),
                        horizontal_spacing=0.05, vertical_spacing=0.07,
                        shared_xaxes=True,  # All subplots in a column share the same X range
                        shared_yaxes=True)   # All subplots in a row share the same Y range)

    keys = list(pos.keys())
    all_shapes = []  
    plotted_energies = set() # Track for clean legend
    planned_legend_added = False

    # 4. Grid Plotting Loop
    for i, p in enumerate(keys):
        row = i // ncols + 1
        col = i % ncols + 1

        ndf = selected_df[selected_df['Spot'] == p].set_index('Energy')
        cx, cy = pos[p]
        
        # Add planned spot marker
        fig.add_trace(
            go.Scatter(
                x=[cx], y=[cy],
                mode='markers',
                marker=dict(symbol='cross', color='black', size=10),
                name='planned',
                showlegend=not planned_legend_added
            ),
            row=row, col=col
        )
        planned_legend_added = True

        # Plot actual spot data
# Plot actual spot data
        for e in ndf.index:
            row_data = ndf.loc[[e]] 
            
            # Extract coordinates based on tolerance
            if tolerance == 2:
                x_vals = row_data['x-pos']
                y_vals = row_data['y-pos']
            else:
                x_vals = row_data['x-pos'] - row_data['centre_abs_xpos'] 
                y_vals = row_data['y-pos'] - row_data['centre_abs_ypos'] 
            
            # Format ADate for the hover label
            # We use .dt.strftime to make it look clean (e.g., 2022-04-21)
            date_strings = row_data['ADate'].dt.strftime('%Y-%m-%d %H:%M')

            # Logic for clean legend
            show_this_legend = False
            if e not in plotted_energies:
                show_this_legend = True
                plotted_energies.add(e)

            base_hex = colours.get(e, '#000000')
            fill_rgba = f"rgba({','.join([str(int(base_hex[i:i+2], 16)) for i in (1, 3, 5)])}, 0.5)"

            fig.add_trace(
                go.Scatter(
                    x=x_vals,
                    y=y_vals,
                    mode='markers',
                    marker=dict(
                        symbol=markers_plotly.get(e, 'circle'),
                        size=10,
                        color=fill_rgba,      # Transparent color fill
                        line=dict(
                            color='black',    # Solid black outline
                            width=1           # Thin outline for precision
                        )
                    ),
                    name=str(e),
                    legendgroup=str(e),
                    showlegend=show_this_legend,
                    customdata=date_strings,
                    hovertemplate=(
                        "<b>Energy: %{fullData.name} MeV</b><br>" +
                        "X: %{x:.2f}<br>" +
                        "Y: %{y:.2f}<br>" +
                        "Date: %{customdata}<extra></extra>"
                    )
                ),
                row=row, col=col
            )

        # 5. Tolerance Shapes (Set to layer='below')
        all_shapes.extend([
            dict(
                type='circle',
                layer='below', 
                xref=f'x{i+1}', yref=f'y{i+1}',
                x0=cx - tolerance, y0=cy - tolerance,
                x1=cx + tolerance, y1=cy + tolerance,
                line=dict(color='#DBB40C'),
                fillcolor='#F5F5DC',
                opacity=0.5,
            ),
            dict(
                type='rect',
                layer='below',
                xref=f'x{i+1}', yref=f'y{i+1}',
                x0=cx - tolerance, y0=cy - tolerance,
                x1=cx + tolerance, y1=cy + tolerance,
                line=dict(color='#DBB40C', width=1)
            )
        ])
        
        # Invert Y-axis for radiotherapy coordinate systems
        fig.update_xaxes(range=[cx - b, cx + b], row=row, col=col)
        fig.update_yaxes(range=[cy + b, cy - b], scaleanchor=f"x{i+1}", row=row, col=col)

    # 6. Final Layout
    fig.update_layout(
        shapes=all_shapes,
        title=title,
        height=300 * nrows,
        width=300 * ncols,
        legend=dict(x=1.02, y=0.5, traceorder="normal"),
        margin=dict(t=100, l=50, r=50, b=50),
        template="plotly_white"
    )

    return fig

In [8]:
fig = plot_spot_grid_plotly(df = sub_df, gantry = "Gantry 1", device = "XRV-3000", tolerance = 2, n_months =12)

fig

NameError: name 'plot_spot_grid_plotly' is not defined

## SBE

In [ ]:

def plot_spot_grid_plotly_sbe(df, gantry, device, tolerance, n_months):
    """ plot the data in grids. """

    # 1. Data Preparation
    df['ADate'] = pd.to_datetime(df['ADate'])
    start_date = pd.Timestamp.today() - pd.DateOffset(months=n_months)
    selected_df = df[(df["MachineName"]==gantry) & (df["Device"] == device) & (df['ADate'] >= start_date)].copy()

    if selected_df.empty:
        print(f"No data found for {gantry} using {device} in the last {n_months} months.")
        return None

    # 2. Dynamic Color Palette (HLS)
    # Sorting ensures the color gradient follows the energy levels
    ens = sorted(list(selected_df["Energy"].unique()), reverse=True)
    hls_palette = sns.color_palette("hls", len(ens)).as_hex()
    colours = dict(zip(ens, hls_palette))

    # 3. Plot Configuration
    if tolerance == 1:
        b = 2.5
        title = f"Relative spot positions ({tolerance} mm tolerance)"
    else:
        b = 5
        title = f"Absolute spot positions ({tolerance} mm tolerance)"
        
    if device == 'XRV-4000':
        pos = pred_xrv4000
        nrows, ncols = 5, 3
    elif device == 'XRV-3000':
        pos = pred_xrv3000
        nrows, ncols = 3, 3
    
    fig = make_subplots(rows=nrows, cols=ncols,
                        subplot_titles=list(pos.keys()),
                        horizontal_spacing=0.05, vertical_spacing=0.07,
                        shared_xaxes=True,  # All subplots in a column share the same X range
                        shared_yaxes=True)   # All subplots in a row share the same Y range)

    keys = list(pos.keys())
    all_shapes = []  
    plotted_energies = set() # Track for clean legend
    planned_legend_added = False
    
    # 4. Add filtering from drop down boxes and month slider
    # Functions to enable automatically updating graph filtering from drop-down menus
    def visible_for_g(target_g): # Gantry selection
        return [
            True if g == target_g else "legendonly"
            for g in ["Gantry 1", "Gantry 2", "Gantry 3", "Gantry 4"]
        ]
    
    def visible_for_ga(target_ga): # Gantry Angle selection
        return [
            True if ga == target_ga else "legendonly"
            for ga in ["0", "90", "180", "270"]
        ]
    
    def visible_for_e(target_e): # Energy selection
        return [
            True if e == target_e else "legendonly"
            for e in ["70", "100", "150", "200", "270"]
        ]

    # Define Gantry Drop-down menu
    g_button = []
    for g in ["Gantry 1", "Gantry 2", "Gantry 3", "Gantry 4"]:
            g_button.append(
                dict(
                    label=f"{g}",
                    method="update",
                    args=[{"visible": visible_for_g(g)}]
                    )
            )

    # Define Gantry Angle drop-down menu
    ga_button = []
    for ga in ["0", "90", "180", "270"]:
            ga_button.append(
                dict(
                    label=f"{ga}",
                    method="update",
                    args=[{"visible": visible_for_ga(ga)}]
                    )
            )

    # Define Energy drop-down menu
    e_button = []
    for e in ["70", "100", "150", "200", "270"]:
            e_button.append(
                dict(
                    label=f"{e} MeV",
                    method="update",
                    args=[{"visible": visible_for_e(e)}]
                    )
            )

    fig.update_layout(
        updatemenus=[
            # Update graph to reflect Gantry drop down box value
            dict(
                type="dropdown",
                direction="down",
                x=0.05,
                y=1.06,
                xanchor="left",
                yanchor="top",
                showactive=True,
                buttons=g_button
            ),

            # Update graph to reflect Gantry Angle drop down box value
            dict(
                type="dropdown",
                direction="down",
                x=0.29,
                y=1.06,
                xanchor="left",
                yanchor="top",
                showactive=True,
                buttons=ga_button
            ),

            # Update graph to reflect Energy drop down box value
            dict(
                type="dropdown",
                direction="down",
                x=0.49,
                y=1.06,
                xanchor="left",
                yanchor="top",
                showactive=True,
                buttons=e_button
            )
        ]
    ),

    fig.update_layout(
        annotations=[
            dict(text="Gantry:", x=0.0, xref="paper", y=1.02, yref="paper", align = "left", showarrow=False),
            dict(text="Angle:", x=0.24, xref="paper", y=1.02, yref="paper", align="left", showarrow=False),
            dict(text="Energy:", x=0.44, xref="paper", y=1.02, yref="paper", align="left", showarrow=False)
        ])

    # 5. Grid Plotting Loop
    for i, p in enumerate(keys):
        row = i // ncols + 1
        col = i % ncols + 1
        cx, cy = pos[p]
        ndf = selected_df[selected_df['Spot'] == p].set_index('Energy')
        
        # Add planned spot marker
        fig.add_trace(
            go.Scatter(
                x=[cx], y=[cy],
                mode='markers',
                marker=dict(symbol='cross', color='black', size=5),
                name='planned',
                showlegend=not planned_legend_added
            ),
            row=row, col=col
        )
        planned_legend_added = True

        # Add most recent spot marker...


        # Plot historical spot data
        for e in ndf.index:
            row_data = ndf.loc[[e]] 
            
            # Extract coordinates based on tolerance
            if tolerance == 2:
                x_vals = row_data['x-pos']
                y_vals = row_data['y-pos']
            else:
                x_vals = row_data['x-pos'] - row_data['centre_abs_xpos'] 
                y_vals = row_data['y-pos'] - row_data['centre_abs_ypos'] 
            
            # Format ADate for the hover label
            # We use .dt.strftime to make it look clean (e.g., 2022-04-21)
            date_strings = row_data['ADate'].dt.strftime('%Y-%m-%d %H:%M')

            # Logic for clean legend
            show_this_legend = False
            if e not in plotted_energies:
                show_this_legend = True
                plotted_energies.add(e)


            # Plot all data seperately, most recent = red cross then increasing transparency for previous results
            base_hex = colours.get(e, '#000000')
            fill_rgba = []; fill_rgba.append("rgba(256,0,0,1)");
            symbol_array = []; symbol_array.append("x");
            
            for n in range(1, x_vals.size):
                fill_rgba.append(f"rgba({','.join([str(int(base_hex[i:i+2], 16)) for i in (1, 3, 5)])}, {0.1* (1 - (n/x_vals.size))})")
                symbol_array.append(markers_plotly.get(e, 'circle'));
            
            fig.add_trace(
                go.Scatter(
                    x=x_vals,
                    y=y_vals,
                    mode='markers',
                    marker=dict(
                        symbol=symbol_array,
                        size=7,
                        color=fill_rgba,
                    ),
                    name=str(e),
                    legendgroup=str(e),
                    showlegend=show_this_legend,
                    customdata=date_strings,
                    hovertemplate=(
                        "<b>Energy: %{fullData.name} MeV</b><br>" +
                        "X: %{x:.2f}<br>" +
                        "Y: %{y:.2f}<br>" +
                        "Date: %{customdata}<extra></extra>"
                    )
                ),
                row=row, col=col,
                
            )

        # 6. Tolerance Shapes (Set to layer='below')
        all_shapes.extend([
            dict(
                type='circle',
                layer='below', 
                xref=f'x{i+1}', yref=f'y{i+1}',
                x0=cx - tolerance, y0=cy - tolerance,
                x1=cx + tolerance, y1=cy + tolerance,
                line=dict(color='#DBB40C'),
                fillcolor='#F5F5DC',
                opacity=0.5,
            ),
            dict(
                type='rect',
                layer='below',
                xref=f'x{i+1}', yref=f'y{i+1}',
                x0=cx - tolerance, y0=cy - tolerance,
                x1=cx + tolerance, y1=cy + tolerance,
                line=dict(color='#DBB40C', width=1)
            )
        ])
        
        # Invert Y-axis for radiotherapy coordinate systems
        fig.update_xaxes(range=[cx - b, cx + b], row=row, col=col)
        fig.update_yaxes(range=[cy + b, cy - b], scaleanchor=f"x{i+1}", row=row, col=col)

    # 7. Create and add slider
    months = []
    for i in range(n_months):
        month = dict(
            method="update",
            args=[{"visible": [True] * len(fig.data)}  # layout attribute
        ])
        month["args"][0]["visible"][i] = True  # Toggle i'th trace to "visible"
        months.append(month)

    # Define sliders for date range selection
    sliders = [dict(
    active=10,
    currentvalue={"prefix": "Previous "},
    pad={"t": 50},
    steps=months
    )]

    fig.update_layout(
        sliders=sliders
    )

    i=0
    for step in fig.layout.sliders[0].steps:
        step['label'] = f"{i} Months"
        i += 1
    
    # 8. Final Layout
    fig.update_layout(
        shapes=all_shapes,
        title=title,
        height=300 * nrows,
        width=300 * ncols,
        margin=dict(t=100, l=50, r=50, b=50),
        legend=dict(x=1.02, y=0.5, traceorder="normal"),
        template="plotly_white"
    )

    return fig

In [10]:
fig = plot_spot_grid_plotly_sbe(df = sub_df, gantry = "Gantry 1", device = "XRV-3000", tolerance = 2, n_months =12)

fig